
Environment Specifications:

Disk size: 2,500 GB

CPUs: 4, RAM: 26

GPU: NVIDIa Tesla T4: 1

In [ ]:
import os 
BUCKET = os.environ['WORKSPACE_BUCKET']
print(BUCKET)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
%env CUDA_VISIBLE_DEVICES=0

In [ ]:
%%bash
cd /home/jupyter/workspaces/multimodalaifordiseaseriskpredictioninallofusparticipantsv8
GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/broadinstitute/ml4h.git
cd ml4h
GIT_LFS_SKIP_SMUDGE=1 git switch -f sf_trans
GIT_LFS_SKIP_SMUDGE=1 git pull
pip install .

In [ ]:
! pip install nvidia-cudnn-cu11==9.8.0.87

In [ ]:
import os 
BUCKET = os.environ['WORKSPACE_BUCKET']
print(BUCKET)

import sys
sys.path.append('/home/jupyter/')

import os
import sys
import pandas as pd
import numpy as np
import random
import glob
import h5py
import matplotlib.pyplot as plt
import json
import datetime

%matplotlib inline


# Package Installation

# Copying Files

In [ ]:
! gsutil -m cp -r $BUCKET/multimodal_ml_baselines/transformer/input/incident /home/jupyter/workspaces/multimodalaifordiseaseriskpredictioninallofusparticipantsv8/

In [ ]:
! gsutil -m cp $BUCKET/multimodal_ml_baselines/data/splits_incident_stratified_age_sex.json /home/jupyter/workspaces/multimodalaifordiseaseriskpredictioninallofusparticipantsv8/

In [ ]:
with open(f'./splits_incident_stratified_age_sex.json') as f:
    splits = json.load(f)

In [ ]:
from ml4h.recipes import train_transformer_on_parquet, train_transformer_on_parquet_fast
from ml4h.arguments import parse_args

In [ ]:
cur_phecode = 'CV_401'
df = pd.read_parquet(f'/home/jupyter/workspaces/multimodalaifordiseaseriskpredictioninallofusparticipantsv8/incident/{cur_phecode}__incident__5to1095.75days.pq')
df['delta_norm'] = (df.delta - df.delta.mean()) / df.delta.std()
df[cur_phecode] = df[f'{cur_phecode}__bp30']
df.to_parquet(f'/home/jupyter/workspaces/multimodalaifordiseaseriskpredictioninallofusparticipantsv8/incident/{cur_phecode}_norm.pq')

In [ ]:
df[cur_phecode].value_counts()

In [ ]:
df.head()

In [ ]:
train_ids = splits[cur_phecode]['train'][:-1000]
valid_ids = splits[cur_phecode]['train'][-1000:]
test_ids = splits[cur_phecode]['test']
pd.Series(train_ids, name="person_id").to_csv(f"train_person_ids_incident_{cur_phecode}.csv", index=False)
pd.Series(valid_ids, name="person_id").to_csv(f"valid_person_ids_incident_{cur_phecode}.csv", index=False)
pd.Series(test_ids, name="person_id").to_csv(f"test_person_ids_incident_{cur_phecode}.csv", index=False)

In [ ]:
import datetime
import sys
now_string = datetime.datetime.now().strftime('%Y_%m_%d')
sys.argv = ['train', 
            '--transformer_input_file', 
            f'/home/jupyter/workspaces/multimodalaifordiseaseriskpredictioninallofusparticipantsv8//incident/{cur_phecode}_norm.pq', 
            #f"/home/jupyter/workspaces/multimodalaifordiseaseriskpredictioninallofusparticipantsv8/prevalent/{cur_phecode}_rechunked_10k.pq",
            '--learning_rate', '0.00002',
            '--inspect_model',
            '--id', f'transformer_prevalent_{cur_phecode}_v{now_string}',
            '--output_folder', './runs/incident_3year/',
            '--training_steps', '256', '--validation_steps', '192', '--test_steps', '300',
            '--epochs', '48', '--patience', '64', '--batch_size', '16',
            '--input_numeric_columns', 'delta_norm',
            '--latent_dimensions', '10',
            '--target_binary_columns', cur_phecode,
            '--transformer_latent_embed', '128', 
            '--transformer_scalar_embed', '2',
            '--attention_heads', '4', 
            '--transformer_dropout_rate', '0.3',
            '--transformer_max_size', '1000',
            '--transformer_size', '128',
            '--transformer_layers', '4',
            '--group_column', 'person_id', 
            '--sort_column', 'date', 
            '--train_csv', f'train_person_ids_incident_{cur_phecode}.csv',
            '--valid_csv', f'valid_person_ids_incident_{cur_phecode}.csv',
            '--test_csv', f'test_person_ids_incident_{cur_phecode}.csv',
           # '--model_file', './runs/incident_3year//transformer_prevalent_NS_333_v2026_03_25/transformer_prevalent_NS_333_v2026_03_25.keras',
    ]


args = parse_args()

In [ ]:
train_transformer_on_parquet_fast(args)

In [ ]:
phecodes = """EM_202_2
MB_286_2
MB_288
MB_290
MB_304
CV_425
MS_707
MS_710
CV_413
CV_413_1
ID_059
DE_668
EM_204
EM_204_1
EM_232_4
DE_681
CV_438
CV_444
CV_439
CA_139
CA_142_1
CA_142
CA_144
BI_160
SO_392_1
SO_394_2
SO_396_8
NS_331
NS_333
NS_333_3
NS_333_6""".split('\n')

In [ ]:
min_days = 5
for cur_phecode in phecodes:
    print(cur_phecode)
    df = pd.read_parquet(f'/home/jupyter/workspaces/multimodalaifordiseaseriskpredictioninallofusparticipantsv8/incident/{cur_phecode}__incident__5to1095.75days.pq')
    df['delta_norm'] = (df.delta - df.delta.mean()) / df.delta.std()
    df[cur_phecode] = df[f'{cur_phecode}__bp30']
    
    daycount_df = df[['person_id','date']].groupby('person_id').agg('count')
    pids = daycount_df.loc[daycount_df['date']>=min_days].index.drop_duplicates().to_list()
    df = df.loc[df['person_id'].isin(pids)]
    df.to_parquet(f'/home/jupyter/workspaces/multimodalaifordiseaseriskpredictioninallofusparticipantsv8/incident/{cur_phecode}_norm.pq') 

    train_ids = splits[cur_phecode]['train'][:-1000]
    valid_ids = splits[cur_phecode]['train'][-1000:]
    test_ids = splits[cur_phecode]['test']
    pd.Series(train_ids, name="person_id").to_csv(f"train_person_ids_incident_{cur_phecode}.csv", index=False)
    pd.Series(valid_ids, name="person_id").to_csv(f"valid_person_ids_incident_{cur_phecode}.csv", index=False)
    pd.Series(test_ids, name="person_id").to_csv(f"test_person_ids_incident_{cur_phecode}.csv", index=False)
    
    now_string = datetime.datetime.now().strftime('%Y_%m_%d')
    sys.argv = ['train', 
                '--transformer_input_file', 
                f'/home/jupyter/workspaces/multimodalaifordiseaseriskpredictioninallofusparticipantsv8//incident/{cur_phecode}_norm.pq', 
                #f"/home/jupyter/workspaces/multimodalaifordiseaseriskpredictioninallofusparticipantsv8/prevalent/{cur_phecode}_rechunked_10k.pq",
                '--learning_rate', '0.00002',
                '--inspect_model',
                '--id', f'transformer_incident_{cur_phecode}_v{now_string}',
                '--output_folder', './runs/incident_3year/',
                '--training_steps', '256', '--validation_steps', '60', '--test_steps', '185',
                '--epochs', '64', '--patience', '64', '--batch_size', '16',
                '--input_numeric_columns', 'delta_norm',
                '--latent_dimensions', '10',
                '--target_binary_columns', cur_phecode,
                '--transformer_latent_embed', '128', 
                '--transformer_scalar_embed', '2',
                '--attention_heads', '4', 
                '--transformer_dropout_rate', '0.3',
                '--transformer_max_size', '1000',
                '--transformer_size', '128',
                '--transformer_layers', '5',
                '--group_column', 'person_id', 
                '--sort_column', 'date', 
                '--train_csv', f'train_person_ids_incident_{cur_phecode}.csv',
                '--valid_csv', f'valid_person_ids_incident_{cur_phecode}.csv',
                '--test_csv', f'test_person_ids_incident_{cur_phecode}.csv',
               # '--model_file', './runs/incident_3year//transformer_prevalent_NS_333_v2026_03_25/transformer_prevalent_NS_333_v2026_03_25.keras',
        ]


    args = parse_args()
    train_transformer_on_parquet_fast(args)

In [ ]:
df = pd.read_csv('all_phe_results_concat.csv')

df.head()

In [ ]:

for phecode in df.phecode.unique().tolist():

    m = df[df.phecode == phecode].auroc.max()
    print(f'{phecode} : {m}')

In [ ]:
df.sort_values(by='phecode').groupby('phecode').head(50)

In [ ]:
import math
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

root = Path(f"./runs//")
png_files = sorted(root.rglob("*.png"))

MAX_PIXELS = 150_000_000  # safe threshold you choose

print(f"Found {len(png_files)} PNG files")

shown = 0
skipped = 0

for png in png_files:
    try:
        with Image.open(png) as im:
            w, h = im.size
            npx = w * h
#             if 'heatmap_performance_auroc' not in str(png):
#                 continue
            if npx > MAX_PIXELS:
                print(f"SKIP (too large {w}x{h}={npx:,} px): {png}")
                skipped += 1
                continue

            # load and display
            im.load()
            plt.figure(figsize=(6, 6))
            plt.imshow(im)
            plt.title(png.as_posix(), fontsize=8)
            plt.axis("off")
            plt.show()
            shown += 1

    except Exception as e:
        print(f"ERROR reading {png}: {e}")

print(f"\nShown: {shown}, Skipped: {skipped}")

In [ ]:
transformer_prevalent_SS_834_v2026_03_06

In [ ]:
from pathlib import Path
import json

base_path = Path("./runs/2year_window")

json_data = []

for file_path in base_path.rglob("metrics_*.json"):
    with open(file_path, "r") as f:
        json_data.extend(json.load(f))

print(f"Loaded {len(json_data)} files.")

In [ ]:
df2 = pd.DataFrame(json_data)
df2.head()

In [ ]:
df[df.Metric=='auROC'].head(10)

## Copying QC participants without EHR data

In [ ]:
df = pd.read_csv('all_phe_results_concat.csv')
phecodes = [ 'CV_417_3', 'CV_404_2', 'CV_401','MB_286', 'MB_288', 'EM_236_1',  'NS_333_1']
for phecode in phecodes:

    m = df[df.phecode == phecode].c_index.max()
    print(f'{phecode} Baseline : {m} Transformer: {df2[(df2.Metric=="auROC")&(df2.Task==phecode)].Score.iloc[0]}')

In [ ]:
results = []

for phecode in phecodes:
    baseline = df[df.phecode == phecode].c_index.max()
    transformer = df2[
        (df2.Metric == "auROC") &
        (df2.Task == phecode)
    ].Score.iloc[0]

    results.append({
        "phecode": phecode,
        "Baseline": baseline,
        "Transformer": transformer
    })

plot_df = pd.DataFrame(results)

In [ ]:
x = np.arange(len(plot_df))
width = 0.35

plt.figure(figsize=(10, 6))

plt.bar(x - width/2, plot_df["Baseline"], width, label="Best Baseline")
plt.bar(x + width/2, plot_df["Transformer"], width, label="Transformer")

plt.xticks(x, plot_df["phecode"], rotation=45)
plt.ylabel("Performance")
plt.title("Baseline vs Transformer Performance by Phecode")
plt.legend()

plt.tight_layout()
plt.show()

## Checking copied numbers

In [ ]:
! gsutil du $BUCKET/ingested_data_without_ehr/all_qc/ | wc -l
! ls -1 /home/jupyter/workspaces/multimodalaifordiseaseriskpredictioninallofusparticipantsv8/ingested_fitbit_data | wc -l

# Python Imports

In [ ]:
import ml4h
from ml4h.arguments import parse_args
from ml4h.TensorMap import TensorMap, Interpretation
from ml4h.models.train import train_model_from_generators
from ml4h.models.model_factory import make_multimodal_multitask_model

from ml4ht.data.sample_getter import DataDescriptionSampleGetter

import ml4h_fitbit_data_definitions
from ml4h_fitbit_data_definitions import ECGDataDescription, ECGDayDataDescription, ECGWeekDataDescription, DataFrameDataDescription, TensorMapDataLoader2
from ml4h_fitbit_data_definitions import dataframe_data_description_from_tensor_map, dataframe_mixed_data_description_from_tensor_map, one_hot_sex, one_hot_afib, standardize_by_sample_hr, save_to_google_cloud

from ml4h.arguments import parse_args
from ml4h.TensorMap import TensorMap, Interpretation
from ml4h.models.train import train_model_from_generators
from ml4h.models.model_factory import make_multimodal_multitask_model
from ml4ht.data.data_loader import SampleGetterIterableDataset, shuffle_get_epoch

import gcsfs

fs = gcsfs.GCSFileSystem()

# Training

## Definitions

In [ ]:
########################## Label to Use and Settings ##############################
lbl_field = 'autoencoder_no_ehr_weekly' # 'autoencoder_no_ehr' # 'autoencoder_finetune' # 'autoencoder_no_ehr'  # Choose from: 'beta_blocker_ae', 'beta_blocker', 'autoencoder', 'anemia', 'obesity', 'afib', 'cad', 'hf', 'syncope', 'arrhythmia' (later to separate to 3 different labels)
# 'arrhythmia_only', 'tachycardia', 'bradycardia', 'stroke', 'stroke_mgb_def'
incident_cond = False # Incident disease prediction (True) or prevalent disease classification (False)

bb_blocker_subj_only = False  # Training a model between days with beta-blockers and without, only for subjects that received beta-blockers at some point
# - TODO - change to a more general variable - e.g. daily level cond / drug / ... 

is_contrastive = False  # for autoencoder - use contrastive loss in training

artificially_balanced_train_set = True  # duplicate positive samples in train set to have balanced class numbers

load_splits = False

hr_len = 1440  # number of HR samples per day (number of minutes)
hr_len_thresh_all = 1200  # threshold for number of non-zero HR minutes - only days with more than 'hr_len_thresh_all' non-zero HR samples will be used
hr_len_thresh = hr_len_thresh_all
if lbl_field=='beta_blocker_ae':
    hd5_folder = '/home/jupyter/workspaces/multimodalaifordiseaseriskpredictioninallofusparticipantsv8/runs/hr_autoencoder_v2025_02_23/model_output__embeddings/'
else:
    hd5_folder = '/home/jupyter/workspaces/multimodalaifordiseaseriskpredictioninallofusparticipantsv8/ingested_fitbit_data/'
######################################################################

### Label functions and other general data functions

In [ ]:
### Label functions:

def beta_blocker_label(row):
    if (int(row['sample_id']) in bb_subj_ids):
        return 1 # subject with beta blockers given in days with valid fitbit HR data (missingness lower than threshold)
    elif (int(row['sample_id']) in bb_subj_ids_all):
        return 2 # subjects with beta blocker records but NOT given in days with valid fitbit HR data (missingness lower than threshold)
    else:
        return 0 # No beta blockers recorded in EHR

def cond_label(row):
    # 0 no afib, 1 prevalent, 2 incident
    if (row['condition_first_date'] is not np.nan) and (row['condition_first_date'] is not pd.to_datetime('NaT')):
        if row['condition_first_date'] <= pd.to_datetime(row['first_hr_date']):
            return 1  # prevalent
        elif (row['condition_last_date'] is not np.nan) and (row['condition_last_date'] is not pd.to_datetime('NaT')):
            if pd.to_datetime(row['first_hr_date']) <= row['condition_last_date']:
                return 2  # incident - in-between first and last condition dates
            else:
                return 0 # no condition
        else:
            return 2  # incident - no info about last condition dates
    else:
        return 0 # no condition

### Other data functions:

def make_zscore(mu, std):
    def zscore(x):
        return (x-mu) / (1e-8+std)
    return zscore

## Load and organize meta-data

In [ ]:
beta_blockers_df = None

if (lbl_field == 'autoencoder_no_ehr') or (lbl_field == 'autoencoder_no_ehr_weekly'):
    dem_df = pd.read_csv(f'{BUCKET}/fitbit_data/fitbit_demographics_no_ehr_v8.tsv',sep='\t')
    fitbit_hr_dates = pd.read_csv(f'{BUCKET}/fitbit_data/fitbit_hr_min_level_dates_person_id_by_person_id_no_ehr_v8.tsv', sep='\t')
    dem_hr_date_df = dem_df.merge(fitbit_hr_dates.rename(columns={"date": "first_hr_date"}), on='person_id', how='left').drop_duplicates()
    
    dem_hr_date_df.loc[dem_hr_date_df['first_hr_date'].isnull(),'first_hr_date'] = pd.to_datetime("2025-01-01", format="%Y-%m-%d")
else:
    dem_df = pd.read_csv(f'{BUCKET}/fitbit_data/fitbit_demographics_v8.tsv',sep='\t')
    fitbit_hr_dates = pd.read_csv(f'{BUCKET}/fitbit_data/fitbit_hr_min_level_dates_person_id_by_person_id_v8.tsv', sep='\t')
    dem_hr_date_df = dem_df.merge(fitbit_hr_dates.rename(columns={"date": "first_hr_date"}), on='person_id').drop_duplicates()
    
print(f'All fitbit with EHR and HR: {len(fitbit_hr_dates)}')
print(f'Merged fitbit EHR HR + Ingested: {len(dem_hr_date_df)}')
print(f'All ingested: {len(dem_df)}')


    
if lbl_field in ['anemia', 'obesity', 'afib', 'cad', 'hf', 'syncope', 'arrhythmia', 'arrhythmia_only', 'tachycardia', 'bradycardia', 'stroke', 'stroke_mgb_def']:
    cond_info_df = pd.read_parquet(f'{BUCKET}/fitbit_data/fitbit_{lbl_field}_df_v8.pq')
    meta_df = dem_hr_date_df.merge(cond_info_df, on='person_id', how='left')
#     meta_df = dem_df.merge(cond_info_df, on='person_id', how='left')
    print(f'Merged fitbit, ingested and {lbl_field}: {len(meta_df)}')
    
    lbl_function = lambda x: cond_label(x)
    lbl_function_descriptions = [f'No {lbl_field}', f'Prevalent {lbl_field}', f'Incident {lbl_field}']
    if incident_cond:
        pos_label = 2  # for incident disease prediction
    else:
        pos_label = 1  # for prevalent disease classification
    
elif lbl_field == 'beta_blocker':
    beta_blockers_df = pd.read_csv(f'{BUCKET}/fitbit_data/fitbit_betablockers_all_v8.tsv', sep='\t')
    bb_avg_hr = pd.read_parquet(f'{BUCKET}/fitbit_data/fitbit_betablockers_avg_hr_drug_dose_v8.pq')
    
    bb_subj_ids_all = np.unique(beta_blockers_df['person_id'].to_list())
    bb_subj_ids = np.unique(bb_avg_hr.loc[(bb_avg_hr['non_zero_num']>hr_len_thresh_all) & (bb_avg_hr['bb_binary']==1),'person_id'].to_list())
    
    meta_df = dem_hr_date_df
#     meta_df = dem_df
    
    lbl_function = lambda x: beta_blocker_label(x)
    lbl_function_descriptions = ['No BB+HR', 'BB+HR', 'BB No HR']
    pos_label = 1  # for subjects with beta-blockers in days with HR min-level data

elif lbl_field == 'autoencoder':
    meta_df = dem_hr_date_df
#     meta_df = dem_df
    
    lbl_function = lambda x: x
    lbl_function_descriptions = []
    pos_label = 0

elif lbl_field == 'autoencoder_finetune':
    phecode_df = pd.read_parquet(f'{BUCKET}/all_data_info/'+phecode2classify_fname)
    
    meta_df = pd.merge(dem_hr_date_df,phecode_df, on='person_id', how='left')
        
    lbl_function = lambda x: x
    lbl_function_descriptions = []
    pos_label = 0
    
    missingness_df = pd.read_parquet(f'{BUCKET}/all_data_info/hr_qc_missing_samples_per_date_with_ehr.pq')
    
elif lbl_field == 'autoencoder_no_ehr':
    meta_df = dem_hr_date_df
    
    lbl_function = lambda x: x
    lbl_function_descriptions = []
    pos_label = 0
    
    missingness_df = pd.read_parquet(f'{BUCKET}/all_data_info/hr_qc_missing_samples_per_date_fixed.pq')

elif lbl_field == 'autoencoder_no_ehr_weekly':
    meta_df = dem_hr_date_df
        
    lbl_function = lambda x: x
    lbl_function_descriptions = []
    pos_label = 0
    
    missingness_df = pd.read_parquet(f'{BUCKET}/all_data_info/hr_qc_missing_samples_per_date_fixed.pq')

    missingness_df['day'] = pd.to_datetime(missingness_df['date']).dt.strftime('%a')
    missingness_df['week_start'] = pd.to_datetime(missingness_df['date']) - pd.to_timedelta(pd.to_datetime(missingness_df['date']).dt.weekday, unit='D')
    no_missingness_df = missingness_df.loc[missingness_df['missing_samples']==0]

    weekly_all_df = missingness_df.groupby(['person_id', 'week_start']).agg(dates=('date', list), count=('date', 'count'), days=('day', list))
    weekly_no_missingness_df = no_missingness_df.groupby(['person_id', 'week_start']).agg(dates=('date', list), count=('date', 'count'), days=('day', list)).reset_index()


In [ ]:
# Organize meta-data data frame

meta_df.rename(columns={'first_date': 'condition_first_date', 
                'last_date': 'condition_last_date',
                'num_occurences': 'condition_num_occurences', 
                'num_unique_dates': 'condition_num_unique_dates',
                'condition_concept_ids': 'condition_concept_ids',
                'standard_concept_names': 'condition_standard_concept_names',
                'condition_start_datetimes': 'condition_start_datetimes',
                'condition_end_datetimes': 'condition_end_datetimes'
               }, inplace=True)

meta_df = meta_df.rename(columns = {'person_id':'sample_id','gender':'is_male'})
meta_df = meta_df.set_index('sample_id')
meta_df['sample_id'] = meta_df.index.astype(str)

# meta_df['age'] = ((pd.to_datetime(meta_df['first_hr_date'], format="%Y-%m-%d").dt.date - pd.to_datetime(meta_df['date_of_birth'], format="%Y-%m-%d").dt.date).astype('timedelta64[D]').astype(int))
meta_df['age'] = pd.to_timedelta(pd.to_datetime(meta_df['first_hr_date'], format="%Y-%m-%d").dt.date - pd.to_datetime(meta_df['date_of_birth'], format="%Y-%m-%d").dt.date).dt.days.astype(int)
# meta_df['age'] = pd.to_timedelta(pd.to_datetime(meta_df['first_hr_date'], format="mixed").dt.date - pd.to_datetime(meta_df['date_of_birth'], format="mixed").dt.date).dt.days.astype(int)
# meta_df['age'].apply(lambda x: make_zscore(meta_df['age'].mean(), meta_df['age'].std())(x))

In [ ]:
# Create label in data frame for condition and print subect numbers:
if (lbl_field != 'autoencoder') and (lbl_field != 'autoencoder_no_ehr') and (lbl_field != 'autoencoder_no_ehr_weekly'):
    meta_df[lbl_field] = meta_df.apply(lbl_function, axis=1)
    print_str = [s+f': {sum(meta_df[lbl_field]==i_s)}' for i_s, s in enumerate(lbl_function_descriptions)]
    print(', '.join(print_str))

In [ ]:
print(f"Num. Female: {sum(meta_df['is_male']=='Female')}, Num. Male: {sum(meta_df['is_male']=='Male')}")

### Define train-validation-test splits

In [ ]:
# Split data
split_labels = ['train', 'valid', 'test']
split_probs = [0.75, 0.15, 0.1]  # 75% train, 15% valid, 10% test
np.random.seed(42)
meta_df['splits'] = ''
if lbl_field == 'autoencoder':
    meta_df['splits'] = np.random.choice(split_labels, size=len(meta_df), p=split_probs)
elif lbl_field == 'autoencoder_no_ehr' or lbl_field == 'autoencoder_no_ehr_weekly':
    split_labels = ['train', 'valid']
    split_probs = [0.85, 0.15]
    
    if load_splits:
        splits_folder = 'hr_autoencoder_no_ehr_v2025_05_03'
        splits_df = pd.read_csv(f'{BUCKET}/talsd_runs/{splits_folder}/meta_df_train_valid_splits.csv')
        meta_df = pd.merge(meta_df, splits_df, on='sample_id', how='inner')
    else:
        meta_df['splits'] = None
        meta_df.loc[meta_df['first_hr_date']!=pd.to_datetime("2025-01-01", format="%Y-%m-%d"),'splits'] = np.random.choice(split_labels, size=sum(meta_df['first_hr_date']!=pd.to_datetime("2025-01-01", format="%Y-%m-%d")), p=split_probs)
    #     meta_df['splits'] = np.random.choice(split_labels, size=len(meta_df), p=split_probs)
else:
    meta_df.loc[meta_df[lbl_field]==0,'splits'] = np.random.choice(split_labels, size=sum(meta_df[lbl_field]==0), p=split_probs)
    meta_df.loc[meta_df[lbl_field]==pos_label,'splits'] = np.random.choice(split_labels, size=sum(meta_df[lbl_field]==pos_label), p=split_probs)



if (lbl_field=='beta_blocker') and bb_blocker_subj_only:
#     split_labels = ['train', 'valid', 'test']
#     split_probs = [0.75, 0.15, 0.1]  # 75% train, 15% valid, 10% test
    np.random.seed(42)
    v7_train_part_date = pd.read_csv(f'{BUCKET}/fitbit_data/beta_blockers_v7_training_ids.csv')
    train_ids = bb_avg_hr.loc[bb_avg_hr[['person_id', 'date']].apply(tuple, axis=1).isin(v7_train_part_date[['person_id', 'date']].apply(tuple, axis=1).tolist())].index

    bb_avg_hr['splits'] = ''
    bb_avg_hr.loc[train_ids,'splits'] = 'train'
#     bb_avg_hr.loc[bb_avg_hr['splits']=='','splits'] = np.random.choice(['valid', 'test'], size=sum((bb_avg_hr['bb_binary']==0) & (bb_avg_hr['non_zero_num']>hr_len_thresh)), p=[0.6,0.4])
    bb_avg_hr.loc[(bb_avg_hr['splits']=='') & (bb_avg_hr['bb_binary']==0) & (bb_avg_hr['non_zero_num']>hr_len_thresh),'splits'] = np.random.choice(['valid', 'test'], size=sum((bb_avg_hr['splits']=='') & (bb_avg_hr['bb_binary']==0) & (bb_avg_hr['non_zero_num']>hr_len_thresh)), p=[0.6,0.4])
    bb_avg_hr.loc[(bb_avg_hr['splits']=='') & (bb_avg_hr['bb_binary']==pos_label) & (bb_avg_hr['non_zero_num']>hr_len_thresh),'splits'] = np.random.choice(['valid', 'test'], size=sum((bb_avg_hr['splits']=='') & (bb_avg_hr['bb_binary']==1) & (bb_avg_hr['non_zero_num']>hr_len_thresh)), p=[0.6,0.4])
    
    print(f"DF train:{len(bb_avg_hr[bb_avg_hr.splits=='train'])} valid:{len(bb_avg_hr[bb_avg_hr.splits=='valid'])} test:{len(bb_avg_hr[bb_avg_hr.splits=='test'])}" )
    
    train_ids = list(bb_avg_hr[bb_avg_hr.splits == 'train'].index)
    np.random.shuffle(train_ids)
    valid_ids = list(bb_avg_hr[bb_avg_hr.splits == 'valid'].index)
    np.random.shuffle(valid_ids)
    test_ids = list(bb_avg_hr[bb_avg_hr.splits == 'test'].index)
    np.random.shuffle(test_ids)
    
    print("Sample IDs train:", len(train_ids), "valid:", len(valid_ids), "test:", len(test_ids))
    print(lbl_field+f" train: {len(bb_avg_hr[np.logical_and(bb_avg_hr.splits == 'train', bb_avg_hr['bb_binary']==1)])}, valid: {len(bb_avg_hr[np.logical_and(bb_avg_hr.splits == 'valid', bb_avg_hr['bb_binary']==1)])}, test: {len(bb_avg_hr[np.logical_and(bb_avg_hr.splits == 'test', bb_avg_hr['bb_binary']==1)])}")

elif (lbl_field=='autoencoder'):
    print(f"DF train:{len(meta_df[meta_df.splits=='train'])} valid:{len(meta_df[meta_df.splits=='valid'])} test:{len(meta_df[meta_df.splits=='test'])}" )    
    # train_ids = list(meta_df[meta_df.splits == 'train'].index)
    train_ids = list(meta_df[meta_df.splits == 'train'].index)
    np.random.shuffle(train_ids)
    valid_ids = list(meta_df[meta_df.splits == 'valid'].index)
    np.random.shuffle(valid_ids)
    test_ids = list(meta_df[meta_df.splits == 'test'].index)
    np.random.shuffle(test_ids)

    print("Sample IDs train:", len(train_ids), "valid:", len(valid_ids), "test:", len(test_ids))
#     print(lbl_field+f" train: {len(meta_df[np.logical_and(meta_df.splits == 'train', meta_df[lbl_field]==1)])}, valid: {len(meta_df[np.logical_and(meta_df.splits == 'valid', meta_df[lbl_field]==1)])}, test: {len(meta_df[np.logical_and(meta_df.splits == 'test', meta_df[lbl_field]==1)])}")
elif lbl_field=='autoencoder_no_ehr':
    
    train_person_ids = meta_df['sample_id'].loc[meta_df.splits == 'train'].astype(int).to_list()
    train_ids = list(missingness_df.loc[np.logical_and(missingness_df['person_id'].isin(train_person_ids), missingness_df['missing_samples']==0)].index)
    np.random.shuffle(train_ids)
    
    valid_person_ids = meta_df['sample_id'].loc[meta_df.splits == 'valid'].astype(int).to_list()
    valid_ids = list(missingness_df.loc[np.logical_and(missingness_df['person_id'].isin(valid_person_ids), missingness_df['missing_samples']==0)].index)
    np.random.shuffle(valid_ids)
    
elif lbl_field=='autoencoder_no_ehr_weekly':
    
    train_person_ids = meta_df['sample_id'].loc[meta_df.splits == 'train'].astype(int).to_list()
    train_ids = list(weekly_no_missingness_df.loc[np.logical_and(weekly_no_missingness_df['person_id'].isin(train_person_ids), weekly_no_missingness_df['count']==7)].index)
    np.random.shuffle(train_ids)
    
    valid_person_ids = meta_df['sample_id'].loc[meta_df.splits == 'valid'].astype(int).to_list()
    valid_ids = list(weekly_no_missingness_df.loc[np.logical_and(weekly_no_missingness_df['person_id'].isin(valid_person_ids), weekly_no_missingness_df['count']==7)].index)
    np.random.shuffle(valid_ids)
    
else:  # beta_blockers all subjects, afib, other conditions
    if artificially_balanced_train_set:
        print(f"DF train:{len(meta_df[meta_df.splits=='train'])} valid:{len(meta_df[meta_df.splits=='valid'])} test:{len(meta_df[meta_df.splits=='test'])}" )    

        imbalance_ratio = int(np.floor(len(meta_df[np.logical_and(meta_df[lbl_field] == 0, meta_df.splits == 'train')])/len(meta_df[np.logical_and(meta_df[lbl_field] == pos_label, meta_df.splits == 'train')])))
        if imbalance_ratio > 1:
            train_ids = list(meta_df[np.logical_and(meta_df[lbl_field] == pos_label, meta_df.splits == 'train')].index)*imbalance_ratio
            train_ids += list(meta_df[np.logical_and(meta_df[lbl_field] == 0, meta_df.splits == 'train')].index)
            print(f'Repeated positive label samples {imbalance_ratio} times')
        else:
            train_ids = list(meta_df[meta_df.splits == 'train'].index)
            
        np.random.shuffle(train_ids)
        valid_ids = list(meta_df[meta_df.splits == 'valid'].index)
        np.random.shuffle(valid_ids)
        test_ids = list(meta_df[meta_df.splits == 'test'].index)
        np.random.shuffle(test_ids)

        print("Sample IDs train:", len(train_ids), "valid:", len(valid_ids), "test:", len(test_ids))
        print(lbl_field+f" train: {len(meta_df[np.logical_and(meta_df.splits == 'train', meta_df[lbl_field]==pos_label)])}, valid: {len(meta_df[np.logical_and(meta_df.splits == 'valid', meta_df[lbl_field]==pos_label)])}, test: {len(meta_df[np.logical_and(meta_df.splits == 'test', meta_df[lbl_field]==pos_label)])}")

        
    else:
        print(f"DF train:{len(meta_df[meta_df.splits=='train'])} valid:{len(meta_df[meta_df.splits=='valid'])} test:{len(meta_df[meta_df.splits=='test'])}" )    
        train_ids = list(meta_df[meta_df.splits == 'train'].index)
        np.random.shuffle(train_ids)
        valid_ids = list(meta_df[meta_df.splits == 'valid'].index)
        np.random.shuffle(valid_ids)
        test_ids = list(meta_df[meta_df.splits == 'test'].index)
        np.random.shuffle(test_ids)

        print("Sample IDs train:", len(train_ids), "valid:", len(valid_ids), "test:", len(test_ids))
        print(lbl_field+f" train: {len(meta_df[np.logical_and(meta_df.splits == 'train', meta_df[lbl_field]==pos_label)])}, valid: {len(meta_df[np.logical_and(meta_df.splits == 'valid', meta_df[lbl_field]==pos_label)])}, test: {len(meta_df[np.logical_and(meta_df.splits == 'test', meta_df[lbl_field]==pos_label)])}")

    
    
print(f"Male train: {len(meta_df[np.logical_and(meta_df.splits == 'train', meta_df['is_male']=='Male')])}, valid: {len(meta_df[np.logical_and(meta_df.splits == 'valid', meta_df['is_male']=='Male')])}, test: {len(meta_df[np.logical_and(meta_df.splits == 'test', meta_df['is_male']=='Male')])}")
print(f"Female train: {len(meta_df[np.logical_and(meta_df.splits == 'train', meta_df['is_male']=='Female')])}, valid: {len(meta_df[np.logical_and(meta_df.splits == 'valid', meta_df['is_male']=='Female')])}, test: {len(meta_df[np.logical_and(meta_df.splits == 'test', meta_df['is_male']=='Female')])}")


### Define TensorMaps and Option Picker

In [ ]:

sex_tmap = TensorMap(name='is_male', interpretation=Interpretation.CATEGORICAL, channel_map={'Female': 0, 'Male':1})
age_tmap = TensorMap(name='age', interpretation=Interpretation.CONTINUOUS, shape=(1,))
avg_hr_tmap = TensorMap(name='avg_hr', interpretation=Interpretation.CONTINUOUS, shape=(1,))

if 'autoencoder' in lbl_field:
    cond_tmap = []
else:
    cond_tmap = TensorMap(name=lbl_field, interpretation=Interpretation.CATEGORICAL, \
                             channel_map={'No_'+lbl_field: 0, lbl_field: 1})


sex_dd = dataframe_data_description_from_tensor_map(sex_tmap, meta_df)
age_dd = dataframe_mixed_data_description_from_tensor_map(age_tmap, meta_df, hd5_folder)
avg_hr_dd = dataframe_mixed_data_description_from_tensor_map(avg_hr_tmap, meta_df, hd5_folder)

if bb_blocker_subj_only and lbl_field == 'beta_blocker':
    cond_dd = dataframe_data_description_from_tensor_map(cond_tmap, bb_avg_hr, bb_ppl_only=True)
elif not ('autoencoder' in lbl_field):
    cond_dd = dataframe_data_description_from_tensor_map(cond_tmap, meta_df)

    
## HR tensor maps and data descriptions:    
    
hr_tmap = TensorMap(
    f'hr_{hr_len}_std',
    Interpretation.CONTINUOUS,
    shape=(hr_len, 1),
)

hr_weekly_tmap = TensorMap(
    f'hr_{hr_len}_std',
    Interpretation.CONTINUOUS,
    shape=(hr_len, 7),
)

hr_weekly_2d_tmap = TensorMap(
    f'hr_{hr_len}_std',
    Interpretation.CONTINUOUS,
    shape=(hr_len, 7, 1),
)
    
hr_tmap_one = TensorMap(
    f'hr_{hr_len}_std',
    Interpretation.CONTINUOUS,
    shape=(hr_len, 1),
)

hr_tmap_two = TensorMap(
    f'hr_{hr_len}_std',
    Interpretation.CONTINUOUS,
    shape=(hr_len, 1),
)


def standardize_by_sample_hr(hr, _):
    """Standardize heart rate data to have mean 0 and standard deviation 1"""
    return (hr - np.mean(hr)) / (np.std(hr) + 1e-6) 

if (lbl_field=='beta_blocker') and bb_blocker_subj_only:
    hr_dd = ECGDayDataDescription(
        hd5_folder, 
        name=hr_tmap.input_name(), 
        hr_len=hr_len,  # all HR arrays will be linearly interpolated to be this length
        transforms=[], #[standardize_by_sample_hr],  # these will be applied in order
        hr_len_thresh=hr_len_thresh,
        beta_blocker=True if lbl_field=='beta_blocker' else False,
        beta_blocker_df=bb_avg_hr
    )
elif is_contrastive:
    hr_dd_one = ECGDataDescription(
        hd5_folder, 
        name=hr_tmap_one.input_name(), 
        hr_len=hr_len,  # all HR arrays will be linearly interpolated to be this length
        transforms=[standardize_by_sample_hr],  # these will be applied in order
        hr_len_thresh=hr_len_thresh,
        beta_blocker=True if lbl_field=='beta_blocker' else False,
        contrastive='one',
    )
    hr_dd_two = ECGDataDescription(
        hd5_folder, 
        name=hr_tmap_two.input_name(), 
        hr_len=hr_len,  # all HR arrays will be linearly interpolated to be this length
        transforms=[standardize_by_sample_hr],  # these will be applied in order
        hr_len_thresh=hr_len_thresh,
        beta_blocker=True if lbl_field=='beta_blocker' else False,
        contrastive='two',
    )
elif (lbl_field == 'autoencoder_no_ehr'):
    hr_dd = ECGDayDataDescription(
        hd5_folder, 
        name=hr_tmap.input_name(), 
        hr_len=hr_len,  # all HR arrays will be linearly interpolated to be this length
        transforms=[], #[standardize_by_sample_hr],  # these will be applied in order
        hr_len_thresh=hr_len_thresh,
        beta_blocker=True if lbl_field=='beta_blocker' else False,
        beta_blocker_df=missingness_df
    )
    
elif (lbl_field == 'autoencoder_no_ehr_weekly'):
    hr_tmap = hr_weekly_tmap
    
    hr_dd = ECGWeekDataDescription(
        hd5_folder, 
        name=hr_tmap.input_name(), 
        hr_len=hr_len,  # all HR arrays will be linearly interpolated to be this length
        transforms=[], #[standardize_by_sample_hr],  # these will be applied in order
        hr_len_thresh=hr_len_thresh,
        beta_blocker=True if lbl_field=='beta_blocker' else False,
        beta_blocker_df=weekly_no_missingness_df
    )
    
elif (lbl_field != 'autoencoder'):
    hr_dd = ECGDataDescription(
        hd5_folder, 
        name=hr_tmap.input_name(), 
        hr_len=hr_len,  # all HR arrays will be linearly interpolated to be this length
        transforms=[standardize_by_sample_hr],  # these will be applied in order
        hr_len_thresh=hr_len_thresh,
        beta_blocker=True if lbl_field=='beta_blocker' else False,
        beta_blocker_df=beta_blockers_df,
        condition_df=meta_df[['sample_id','condition_first_date','condition_last_date']],
        incident_cond=incident_cond
    )
else:
    
    hr_dd = ECGDataDescription(
        hd5_folder, 
        name=hr_tmap.input_name(), 
        hr_len=hr_len,  # all HR arrays will be linearly interpolated to be this length
        transforms=[standardize_by_sample_hr],  # these will be applied in order
        hr_len_thresh=hr_len_thresh,
        beta_blocker=True if lbl_field=='beta_blocker' else False,
        beta_blocker_df=beta_blockers_df,
        incident_cond=incident_cond
    )

In [ ]:
def option_picker(sample_id, data_descriptions):
    if (((lbl_field=='beta_blocker_ae') or (lbl_field=='beta_blocker')) and bb_blocker_subj_only) or (lbl_field=='autoencoder_no_ehr') or (lbl_field=='autoencoder_no_ehr_weekly'):
        return {dd: sample_id for dd in data_descriptions}
    
    if is_contrastive:
        hr_dts = hr_dd_one.get_loading_options(sample_id)
    else:
        hr_dts = hr_dd.get_loading_options(sample_id)
    dates = []
    for dt in hr_dts:
        dates.append(dt)

    if len(dates) == 0:
        raise ValueError('No matching dates')
    
#     chosen_dt = np.random.choice(dates)
    chosen_ind = np.random.choice(np.arange(len(dates)))
    chosen_gap = np.random.choice(np.concatenate((-np.arange(min(30, chosen_ind))-1,np.arange(min(30, len(dates)-chosen_ind-1))+1)))
                                  
    if is_contrastive:
        return {dd: {'one': dates[chosen_ind], 'two': dates[chosen_ind+chosen_gap]} for dd in data_descriptions}
    else:
        chosen_dt = dates[chosen_ind]
        return {dd: chosen_dt for dd in data_descriptions}


## Setting up Training - Autoencoder

In [ ]:
# Autoencoder:
import datetime
import sys
now_string = datetime.datetime.now().strftime('%Y_%m_%d')
# is_contrastive = True
if is_contrastive:
    sys.argv = ['train', '--tensors', '/', 
                '--learning_rate', '0.00001',
                '--inspect_model',
                '--id', f'hr_autoencoder_contrastive_v{now_string}',
                '--output_folder', './runs/',
                '--training_steps', '128', '--validation_steps', '32',
                '--epochs', '1000', '--patience', '20', '--batch_size', '4',
                '--encoder_blocks', 'conv_encode', 'conv_encode',
                '--merge_blocks', 'pair',
                '--decoder_blocks', 'conv_decode', 'conv_decode',
                '--activation', 'mish', 
                '--conv_layers', '23', '23',
                '--conv_width', '25',  # was default = 71
                '--dense_blocks', '46', 
                '--block_size', '5',
                '--dense_layers', '256',
                '--dense_normalize', 'layer_norm',
                '--pair_loss', 'contrastive', 
                '--pair_loss_weight', '0.1', 
                '--pair_merge', 'concat']

    
    args = parse_args()
    args.tensor_maps_in = [hr_tmap_one, hr_tmap_two]
    args.tensor_maps_out = [hr_tmap_one, hr_tmap_two]
#     args.tensor_maps_in = [hr_tmap_one]
#     args.tensor_maps_out = [hr_tmap_one]
    args.pairs = [hr_tmap_one, hr_tmap_two]
    
else:
    sys.argv = ['train', '--tensors', '/', 
                '--learning_rate', '0.00001',
                '--inspect_model',
                '--id', f'hr_autoencoder_no_ehr_weekly_v{now_string}',
                '--output_folder', './runs/',
                '--training_steps', '500', '--validation_steps', '32',  # '--training_steps', '1000', '--validation_steps', '32',
                '--epochs', '500', '--patience', '20', '--batch_size', '8',
                '--encoder_blocks', 'conv_encode',
                '--decoder_blocks', 'conv_decode',
                '--activation', 'mish', 
                '--conv_layers', '23', '23',
                '--conv_width', '25',  # was default = 71
                '--dense_blocks', '46', 
                '--block_size', '5',
                '--dense_layers', '256',
                '--dense_normalize', 'layer_norm']

    args = parse_args()
    args.tensor_maps_in = [hr_tmap]
    args.tensor_maps_out = [hr_tmap]
# args.tensor_maps_out = [afib_tmap, sex_tmap, age_tmap]
model, encoders, decoders, merger = make_multimodal_multitask_model(**args.__dict__)

In [ ]:
meta_df[['sample_id','splits']].to_csv(f'./runs/{args.id}/meta_df_train_valid_splits.csv')

! gcloud storage cp runs/$args.id/meta_df_train_valid_splits.csv $BUCKET/talsd_runs/$args.id/meta_df_train_valid_splits.csv

In [ ]:

if is_contrastive:
    hr_dd_out_one = ECGDataDescription(
        hd5_folder, 
        name=hr_tmap.output_name(), 
        hr_len=hr_len,  # all HR arrays will be linearly interpolated to be this length
        transforms=[], #[standardize_by_sample_hr],  # these will be applied in order
        hr_len_thresh=hr_len_thresh,
        contrastive='one',
    )
    hr_dd_out_two = ECGDataDescription(
        hd5_folder, 
        name=hr_tmap.output_name(), 
        hr_len=hr_len,  # all HR arrays will be linearly interpolated to be this length
        transforms=[], #[standardize_by_sample_hr],  # these will be applied in order
        hr_len_thresh=hr_len_thresh,
        contrastive='two',
    )
else:
    if lbl_field == 'autoencoder_no_ehr':
        hr_dd_out = ECGDayDataDescription(
        hd5_folder, 
        name=hr_tmap.output_name(), 
        hr_len=hr_len,  # all HR arrays will be linearly interpolated to be this length
        transforms=[], #[standardize_by_sample_hr],  # these will be applied in order
        hr_len_thresh=hr_len_thresh,
        beta_blocker=True if lbl_field=='beta_blocker' else False,
        beta_blocker_df=missingness_df
        )
    elif lbl_field == 'autoencoder_no_ehr_weekly':
        hr_dd_out = ECGWeekDataDescription(
        hd5_folder, 
        name=hr_weekly_tmap.output_name(), 
        hr_len=hr_len,  # all HR arrays will be linearly interpolated to be this length
        transforms=[], #[standardize_by_sample_hr],  # these will be applied in order
        hr_len_thresh=hr_len_thresh,
        beta_blocker=True if lbl_field=='beta_blocker' else False,
        beta_blocker_df=weekly_no_missingness_df
        )
        
    else:
        hr_dd_out = ECGDataDescription(
        hd5_folder, 
        name=hr_tmap.output_name(), 
        hr_len=hr_len,  # all HR arrays will be linearly interpolated to be this length
        transforms=[], #[standardize_by_sample_hr],  # these will be applied in order
        hr_len_thresh=hr_len_thresh
        )


In [ ]:
if is_contrastive:
    sg_autoencoder = DataDescriptionSampleGetter(
        input_data_descriptions=[hr_dd_one, hr_dd_two],  # what we want the model to use as input data
        output_data_descriptions=[hr_dd_out_one, hr_dd_out_two],
    #     output_data_descriptions=[afib_dd, sex_dd, age_dd],
        option_picker = option_picker,
    )
else:
    sg_autoencoder = DataDescriptionSampleGetter(
        input_data_descriptions=[hr_dd],  # what we want the model to use as input data
        output_data_descriptions=[hr_dd_out],
    #     output_data_descriptions=[afib_dd, sex_dd, age_dd],
        option_picker = option_picker,
    )

    
train_dataset = SampleGetterIterableDataset(sample_ids=list(train_ids), sample_getter=sg_autoencoder, get_epoch=shuffle_get_epoch)
valid_dataset = SampleGetterIterableDataset(sample_ids=list(valid_ids), sample_getter=sg_autoencoder, get_epoch=shuffle_get_epoch)

generate_train = TensorMapDataLoader2(
    batch_size=args.batch_size, input_maps=args.tensor_maps_in, output_maps=args.tensor_maps_out,
    dataset=train_dataset,
    num_workers=0,
    )
generate_valid = TensorMapDataLoader2(
    batch_size=args.batch_size, input_maps=args.tensor_maps_in, output_maps=args.tensor_maps_out,
    dataset=valid_dataset,
    num_workers=0,
    )

In [ ]:
model = train_model_from_generators(
    model, generate_train, generate_valid, args.training_steps, args.validation_steps, args.batch_size, args.epochs,
    args.patience, args.output_folder, args.id, args.inspect_model, args.inspect_show_labels, args.tensor_maps_out,
    save_last_model=args.save_last_model, gcp_bucket=BUCKET.split('//')[-1], gcp_path='talsd_runs/'+args.id,#+'/'+args.id+'.h5',
    log_tensorboard=True,#, log_training_to_file=True,
)
for tm in encoders:
    encoders[tm].save(f'{args.output_folder}{args.id}/encoder_{tm.name}.h5')
for tm in decoders:
    decoders[tm].save(f'{args.output_folder}{args.id}/decoder_{tm.name}.h5')
if merger:
    merger.save(f'{args.output_folder}{args.id}/merger.h5')
    
save_to_google_cloud(args.output_folder, os.path.join(BUCKET.split('//')[1], 'talsd_runs', args.id)+'/', args.id, mode='train')